
# Cento Nuptialis Metrical Stylometry with LambdaG

This notebook analyses the stylistic affinities of Ausonius's *Cento Nuptialis* using
LambdaG authorship verification (uncalibrated) on **metrical number-mapped sentences**.

This is the metrical counterpart to `cento_stylometry.ipynb`, which uses POS-masked sentences.

## Setup

Four model corpora are compared against the cento:
- **vergil**: Vergil, *Georgics* (4 books)
- **aeneid**: Vergil, *Aeneid* (12 books)
- **ausonius**: Ausonius hexameter works (17 works, excluding cento)
- **background**: Ovid, Lucan, Silius Italicus, Valerius Flaccus (50 books)

All corpora contain `*_sentences.metrical.txt` files produced by `scripts/generate_metrical.py`.
Each word is replaced by its prime-encoded metrical position number.


In [ ]:

from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_corpora import Sentence, TwoLevelCorpus
from sentence_corpora.lambdag import LambdaGAV



## Load Corpora

Each directory contains `*_sentences.metrical.txt` files. We parse the
filenames to extract `(author, work)` metadata and build `TwoLevelCorpus` objects.


In [ ]:

def parse_filename(path: Path) -> tuple[str, str]:
    """Parse '<name>_sentences.metrical.txt' into (author, work).

    Filenames from metrical_model/ follow the pattern:
      - ausonius: caesares_sentences.metrical.txt  →  ('caesares', '')
      - vergil: vergil_georgics_1_sentences.metrical.txt  →  ('vergil', 'georgics_1')
      - aeneid: vergil_aeneid_1_sentences.metrical.txt  →  ('vergil', 'aeneid_1')
      - background: ovid_metamorphoses_1_sentences.metrical.txt  →  ('ovid', 'metamorphoses_1')
    """
    name = re.sub(r"_sentences\.metrical\.txt$", "", path.name)
    parts = name.split("_", 1)
    if len(parts) == 1:
        return parts[0], ""
    return parts[0], parts[1]


def load_corpus(data_dir: Path) -> TwoLevelCorpus:
    """Read all *_sentences.metrical.txt files and return a TwoLevelCorpus."""
    sentences = []
    for path in sorted(data_dir.glob("*_sentences.metrical.txt")):
        author, work = parse_filename(path)
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    sentences.append(Sentence(text=line, metadata={"author": author, "work": work}))
    return TwoLevelCorpus(sentences, levels=["author", "work"])


# Load the four model corpora
vergil_corpus = load_corpus(Path("metrical_model/vergil"))
aeneid_corpus = load_corpus(Path("metrical_model/aeneid"))
ausonius_corpus = load_corpus(Path("metrical_model/ausonius"))
background_corpus = load_corpus(Path("metrical_model/background"))

# Load the question text (cento)
cento_path = Path("metrical_model/question/cento_nuptialis_sentences.metrical.txt")
cento_sentences = []
with open(cento_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            cento_sentences.append(Sentence(text=line, metadata={"author": "cento", "work": "nuptialis"}))
cento_corpus = TwoLevelCorpus(cento_sentences, levels=["author", "work"])

print(f"Vergil (Georgics):  {len(vergil_corpus)} sentences, {sum(len(s.text.split()) for s in vergil_corpus)} tokens")
print(f"Aeneid:             {len(aeneid_corpus)} sentences, {sum(len(s.text.split()) for s in aeneid_corpus)} tokens")
print(f"Ausonius:           {len(ausonius_corpus)} sentences, {sum(len(s.text.split()) for s in ausonius_corpus)} tokens")
print(f"Background:         {len(background_corpus)} sentences, {sum(len(s.text.split()) for s in background_corpus)} tokens")
print(f"Cento (question):   {len(cento_corpus)} sentences, {sum(len(s.text.split()) for s in cento_corpus)} tokens")


In [ ]:

vergil_corpus[:10]


In [ ]:

ausonius_corpus[:10]



## Run LambdaG Authorship Verification (Uncalibrated)

For each candidate author, we run a single AV problem:
- **q_corpus**: the cento (question text)
- **kr_corpus**: the model corpus (known reference)
- **known_entity**: the author label in the model corpus

The `LambdaGAV.run_single_av_problem` method computes a raw score indicating
how similar the question text is to the known author. Higher scores = more similar.


In [ ]:

# Run LambdaG AV for each candidate author
# The cento is small (~900 tokens), so we use conservative known/reference token counts
av = LambdaGAV()
rng = np.random.default_rng(42)

results = []

for candidate, candidate_corpus in [("vergil", vergil_corpus), ("ausonius", ausonius_corpus)]:

    # Combine candidate + background into a single KR corpus
    kr_sentences = list(candidate_corpus) + list(background_corpus)
    kr_corpus = TwoLevelCorpus(kr_sentences, levels=["author", "work"])    

    result = av.run_single_av_problem(
        known_entity=candidate,
        q_corpus=cento_corpus,
        kr_corpus=kr_corpus,
        known_tokens=8000,
        reference_tokens=20000,
        order=8,
        num_references=10,
        rng=rng,
    )
    
    results.append({
        "candidate": candidate,
        "score": result.get("score", 0.0),
        "corrected_score_sqrt": result.get("corrected_score_sqrt", 0.0),
        "corrected_score_hapax": result.get("corrected_score_hapax", 0.0),
        "n_unknown": result.get("unknown_sentences_used", 0),
        "unknown_tokens": result.get("unknown_tokens_used", 0),
        "n_known": result.get("known_sentences_used", 0),
        "n_reference": result.get("reference_sentences_used", 0),
        "error": result.get("error", None),
    })
    print(f"  {candidate}: score={results[-1]['score']:.4f}, tokens_q={results[-1]['unknown_tokens']}")

print("\nDone.")



## Results


In [ ]:

df = pd.DataFrame(results)
df = df.sort_values("score", ascending=False)

# Calculate probabilities from both corrected LLRs using prior=0.5
PRIOR = 0.5
df["prob_sqrt"] = df["corrected_score_sqrt"].apply(
    lambda s: LambdaGAV.llr_to_percent_probability(s, PRIOR)
)
df["prob_hapax"] = df["corrected_score_hapax"].apply(
    lambda s: LambdaGAV.llr_to_percent_probability(s, PRIOR)
)

print("LambdaG Metrical Results (uncalibrated, prior=0.5):")
print("=" * 80)
for _, row in df.iterrows():
    print(f"  {row['candidate']:<12}  raw={row['score']:>8.4f}"
          f"  sqrt={row['corrected_score_sqrt']:>8.4f} -> {row['prob_sqrt']:>6.1f}%"
          f"  hapax={row['corrected_score_hapax']:>8.4f} -> {row['prob_hapax']:>6.1f}%"
          f"  (q_tokens={row['unknown_tokens']})")



## Aeneid vs Vergil (Georgics) — Baseline Check


In [ ]:

from typing import Any


def sample_corpus_to_token_count(corpus, target_tokens: int, rng: Any):
    """Sample sentences from corpus until we reach approximately target_tokens."""
    sentences = list(corpus)
    total_tokens = sum(len(s.text.split()) for s in sentences)

    if total_tokens <= target_tokens:
        raise ValueError(
            f"Corpus has only {total_tokens} tokens, which is less than target {target_tokens}"
        )

    shuffled = list(sentences)
    rng.shuffle(shuffled)

    sampled = []
    current_tokens = 0
    for s in shuffled:
        sampled.append(s)
        current_tokens += len(s.text.split())
        if current_tokens >= target_tokens:
            break

    return type(corpus)(sampled, levels=corpus.levels)


print(f"Aeneid corpus: {len(aeneid_corpus)} sentences, {sum(len(s.text.split()) for s in aeneid_corpus)} tokens")


In [ ]:

import ray
from tqdm import tqdm

# Run N iterations: sample ~1079 tokens from Aeneid, compare against Vergil (Georgics) + background
TARGET_TOKENS = 1079
NUM_ITERATIONS = 50

# KR corpus = vergil (Georgics) + background
kr_sentences = list(vergil_corpus) + list(background_corpus)
kr_corpus = TwoLevelCorpus(kr_sentences, levels=["author", "work"])


@ray.remote
def run_aeneid_iteration(i: int) -> dict:
    """Run a single Aeneid vs Vergil AV iteration."""
    iter_rng = np.random.default_rng(42 + i)
    q_corpus = sample_corpus_to_token_count(aeneid_corpus, TARGET_TOKENS, iter_rng)
    actual_tokens = sum(len(s.text.split()) for s in q_corpus)

    av = LambdaGAV()
    result = av.run_single_av_problem(
        known_entity="vergil",
        q_corpus=q_corpus,
        kr_corpus=kr_corpus,
        known_tokens=8000,
        reference_tokens=20000,
        order=8,
        num_references=10,
        rng=iter_rng,
    )

    return {
        "iteration": i,
        "score": result.get("score", 0.0),
        "corrected_score_sqrt": result.get("corrected_score_sqrt", 0.0),
        "corrected_score_hapax": result.get("corrected_score_hapax", 0.0),
        "q_tokens": actual_tokens,
        "n_known": result.get("known_sentences_used", 0),
        "n_reference": result.get("reference_sentences_used", 0),
        "error": result.get("error", None),
    }


ray.init(ignore_reinit_error=True)
print(f"Ray initialized with {ray.available_resources().get('CPU', 4)} CPUs")

tasks = [run_aeneid_iteration.remote(i) for i in range(NUM_ITERATIONS)]

aeneid_results = []
for _ in tqdm(range(len(tasks)), desc="Aeneid AV"):
    ready, tasks = ray.wait(tasks, num_returns=1)
    result = ray.get(ready[0])
    aeneid_results.append(result)

ray.shutdown()
print("\nDone.")


In [ ]:

df_aeneid = pd.DataFrame(aeneid_results)

PRIOR = 0.5
df_aeneid["prob_sqrt"] = df_aeneid["corrected_score_sqrt"].apply(
    lambda s: LambdaGAV.llr_to_percent_probability(s, PRIOR)
)
df_aeneid["prob_hapax"] = df_aeneid["corrected_score_hapax"].apply(
    lambda s: LambdaGAV.llr_to_percent_probability(s, PRIOR)
)

print(f"Aeneid vs Vergil (Georgics) — {NUM_ITERATIONS} iterations, ~{TARGET_TOKENS} tokens each, prior={PRIOR}")
print("=" * 100)
print(f"{'iter':>4}  {'raw':>8}  {'sqrt':>8}  {'prob_sqrt':>10}  {'hapax':>8}  {'prob_hapax':>10}  {'q_tok':>6}")
print("-" * 100)
for _, row in df_aeneid.iterrows():
    print(f"  {int(row['iteration']):>2}  {row['score']:>8.4f}  {row['corrected_score_sqrt']:>8.4f}"
          f"  {row['prob_sqrt']:>9.1f}%  {row['corrected_score_hapax']:>8.4f}"
          f"  {row['prob_hapax']:>9.1f}%  {int(row['q_tokens']):>6}")
print("-" * 100)
print(f"{'mean':>4}  {df_aeneid['score'].mean():>8.4f}  {df_aeneid['corrected_score_sqrt'].mean():>8.4f}"
      f"  {df_aeneid['prob_sqrt'].mean():>9.1f}%  {df_aeneid['corrected_score_hapax'].mean():>8.4f}"
      f"  {df_aeneid['prob_hapax'].mean():>9.1f}%")
print(f"{'std':>4}  {df_aeneid['score'].std():>8.4f}  {df_aeneid['corrected_score_sqrt'].std():>8.4f}"
      f"  {df_aeneid['prob_sqrt'].std():>9.1f}%  {df_aeneid['corrected_score_hapax'].std():>8.4f}"
      f"  {df_aeneid['prob_hapax'].std():>9.1f}%")



## Mosella vs Ausonius — Internal Baseline


In [ ]:

# Build Mosella corpus (q) and Ausonius-minus-Mosella corpus (kr)
mosella_sentences = [s for s in ausonius_corpus if s.work == "mosella"]
ausonius_no_mosella_sentences = [s for s in ausonius_corpus if s.work != "mosella"]

mosella_corpus = TwoLevelCorpus(mosella_sentences, levels=["author", "work"])
ausonius_no_mosella = TwoLevelCorpus(ausonius_no_mosella_sentences, levels=["author", "work"])

# KR corpus = ausonius (without mosella) + background
kr_ausonius_sentences = list(ausonius_no_mosella) + list(background_corpus)
kr_ausonius_corpus = TwoLevelCorpus(kr_ausonius_sentences, levels=["author", "work"])

print(f"Mosella corpus: {len(mosella_corpus)} sentences, {sum(len(s.text.split()) for s in mosella_corpus)} tokens")
print(f"Ausonius (excl. Mosella): {len(ausonius_no_mosella)} sentences, {sum(len(s.text.split()) for s in ausonius_no_mosella)} tokens")
print(f"KR corpus (Ausonius excl. Mosella + background): {len(kr_ausonius_corpus)} sentences")


@ray.remote
def run_mosella_iteration(i: int) -> dict:
    """Run a single Mosella vs Ausonius AV iteration."""
    iter_rng = np.random.default_rng(100 + i)
    q_corpus = sample_corpus_to_token_count(mosella_corpus, TARGET_TOKENS, iter_rng)
    actual_tokens = sum(len(s.text.split()) for s in q_corpus)

    av = LambdaGAV()
    result = av.run_single_av_problem(
        known_entity="ausonius",
        q_corpus=q_corpus,
        kr_corpus=kr_ausonius_corpus,
        known_tokens=4000,
        reference_tokens=20000,
        order=8,
        num_references=10,
        rng=iter_rng,
    )

    return {
        "iteration": i,
        "score": result.get("score", 0.0),
        "corrected_score_sqrt": result.get("corrected_score_sqrt", 0.0),
        "corrected_score_hapax": result.get("corrected_score_hapax", 0.0),
        "q_tokens": actual_tokens,
        "n_known": result.get("known_sentences_used", 0),
        "n_reference": result.get("reference_sentences_used", 0),
        "error": result.get("error", None),
    }


ray.init(ignore_reinit_error=True)

tasks = [run_mosella_iteration.remote(i) for i in range(NUM_ITERATIONS)]

mosella_results = []
for _ in tqdm(range(len(tasks)), desc="Mosella AV"):
    ready, tasks = ray.wait(tasks, num_returns=1)
    result = ray.get(ready[0])
    mosella_results.append(result)

ray.shutdown()
print("\nDone.")


In [ ]:

df_mosella = pd.DataFrame(mosella_results)

df_mosella["prob_sqrt"] = df_mosella["corrected_score_sqrt"].apply(
    lambda s: LambdaGAV.llr_to_percent_probability(s, PRIOR)
)
df_mosella["prob_hapax"] = df_mosella["corrected_score_hapax"].apply(
    lambda s: LambdaGAV.llr_to_percent_probability(s, PRIOR)
)

print(f"Mosella vs Ausonius — {NUM_ITERATIONS} iterations, ~{TARGET_TOKENS} tokens each, prior={PRIOR}")
print("=" * 100)
print(f"{'iter':>4}  {'raw':>8}  {'sqrt':>8}  {'prob_sqrt':>10}  {'hapax':>8}  {'prob_hapax':>10}  {'q_tok':>6}")
print("-" * 100)
for _, row in df_mosella.iterrows():
    print(f"  {int(row['iteration']):>2}  {row['score']:>8.4f}  {row['corrected_score_sqrt']:>8.4f}"
          f"  {row['prob_sqrt']:>9.1f}%  {row['corrected_score_hapax']:>8.4f}"
          f"  {row['prob_hapax']:>9.1f}%  {int(row['q_tokens']):>6}")
print("-" * 100)
print(f"{'mean':>4}  {df_mosella['score'].mean():>8.4f}  {df_mosella['corrected_score_sqrt'].mean():>8.4f}"
      f"  {df_mosella['prob_sqrt'].mean():>9.1f}%  {df_mosella['corrected_score_hapax'].mean():>8.4f}"
      f"  {df_mosella['prob_hapax'].mean():>9.1f}%")
print(f"{'std':>4}  {df_mosella['score'].std():>8.4f}  {df_mosella['corrected_score_sqrt'].std():>8.4f}"
      f"  {df_mosella['prob_sqrt'].std():>9.1f}%  {df_mosella['corrected_score_hapax'].std():>8.4f}"
      f"  {df_mosella['prob_hapax'].std():>9.1f}%")



## Visualisation: Baseline Score Distributions with Cento Markers


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
import mpl_fontkit as fk

fk.install("Roboto Condensed")
plt.rcParams["font.family"] = "Roboto Condensed"

# Extract cento scores from the initial AV run (cell 8)
cento_vergil_score = df.loc[df["candidate"] == "vergil", "score"].values[0]
cento_ausonius_score = df.loc[df["candidate"] == "ausonius", "score"].values[0]

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# --- Aeneid vs Vergil (Georgics) baseline ---
ax = axes[0]
sns.kdeplot(data=df_aeneid, x="score", ax=ax, color="steelblue", linewidth=1.5, fill=True, alpha=0.2, bw_adjust=0.8)
sns.rugplot(data=df_aeneid, x="score", ax=ax, color="steelblue", alpha=0.4, height=0.06)
ax.axvline(cento_vergil_score, color="red", linewidth=1.5, linestyle="--", label=f"Cento vs Vergil ({cento_vergil_score:.3f})")
ax.set_title("Aeneid vs Vergil (Georgics) + Background — Uncalibrated LambdaG Scores", fontsize=13)
ax.set_xlabel("")
ax.legend(fontsize=10, loc="upper right")
sns.despine(ax=ax)

# --- Mosella vs Ausonius baseline ---
ax = axes[1]
sns.kdeplot(data=df_mosella, x="score", ax=ax, color="steelblue", linewidth=1.5, fill=True, alpha=0.2, bw_adjust=0.8)
sns.rugplot(data=df_mosella, x="score", ax=ax, color="steelblue", alpha=0.4, height=0.06)
ax.axvline(cento_ausonius_score, color="red", linewidth=1.5, linestyle="--", label=f"Cento vs Ausonius ({cento_ausonius_score:.3f})")
ax.set_title("Mosella vs Ausonius + Background — Uncalibrated LambdaG Scores", fontsize=13)
ax.set_xlabel("Uncalibrated LambdaG score")
ax.legend(fontsize=10, loc="upper right")
sns.despine(ax=ax)

plt.tight_layout()
plt.show()
